In [15]:
import pandas as pd
import numpy as np
from collections import Counter

In [16]:
data = pd.read_csv(r"../../data/processed/data_vif.csv")

x = data["Risk_Label"].astype(str).str.strip().str.lower()

data["Risk_Label"] = x.map({
    "lowrisk": 0,
    "low risk": 0,
    "low": 0,
    "0": 0,
    "highrisk": 1,
    "high risk": 1,
    "high": 1,
    "1": 1
})

print(data["Risk_Label"].value_counts(dropna=False))

data["Risk_Label"] = data["Risk_Label"].astype(int)

data.head()

Risk_Label
0    3659
1     449
Name: count, dtype: int64


,Date,KODEX 200_Premium,KOSDAQ_return(%),NASDAQ_return(%),Brent Crude Oil_return(%),Gold Spot_return(%),USD/KRW_change(%),KOSPI 200 Volume,return(%),KOSPI 200 lagged_1_return(%),...,GJR_VaR_5_t1,Signal1_Buy,Signal1_Sell,Signal2_Buy,Signal2_Sell,Signal3_Buy,Signal3_Sell,Signal4_Buy,Signal4_Sell,Risk_Label
0,2009-04-17,-0.20,-2.796416,0.157320,0.0,-1.362586,-0.549095,251600000.0,-0.233192,0.303254,...,-2.736404,0,0,0,0,0,0,0,0,0
1,2009-04-20,-0.34,1.668519,-3.953850,0.0,2.234472,0.128139,183300000.0,0.564563,-0.233192,...,-2.640948,0,0,0,0,0,0,0,0,0
2,2009-04-21,-0.14,1.061549,2.191930,0.0,-0.553958,1.998728,204200000.0,-0.197523,0.564563,...,-2.553087,0,0,0,0,0,0,0,0,0
3,2009-04-22,-0.19,2.524236,0.137996,0.0,1.093648,-0.570187,291400000.0,1.408955,-0.197523,...,-2.469280,0,0,0,0,0,0,0,0,0
4,2009-04-23,-0.17,0.818378,0.369276,0.0,1.568707,-0.970084,277400000.0,0.992765,1.408955,...,-2.386329,0,0,0,0,0,1,0,0,0


In [17]:
step=['NASDAQ_return(%)', 'KOSPI 200_OG', 
      'Gold Spot_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 
      'KOSPI 200_PPO_Hist', 'KOSPI 200 lagged_2_return(%)', 'KOSPI 200_RSI14', 
      'USD/KRW_change(%)', 'return(%)', 'Signal1_Buy', 'Signal1_Sell',
     'KODEX 200_Premium', 'VKOSPI_Change(%)', 'Signal4_Buy', 'Signal4_Sell', 'KOSPI 200_BB_width']

mul=['NASDAQ_return(%)', 'Gold Spot_return(%)', 'Brent Crude Oil_return(%)', 
     'KOSPI 200 lagged_1_return(%)', 'VKOSPI_Close', 'return(%)', 'KOSPI 200_ADX14', 
     'VKOSPI_Change(%)', 'KOSDAQ_return(%)', 'KOSPI 200_PPO', 'USD/KRW_change(%)', 
     'KOSPI 200_BB_width', 'Signal2_Buy', 'Signal2_Sell', 'Signal4_Buy', 'Signal4_Sell']

boruta=['KOSPI 200_PPO_Hist', 'KOSPI 200 Volume',
        'KOSPI 200_OG','KOSPI 200_DMI14',
        'GJR_VaR_5_t1','Brent Crude Oil_return(%)',
        'VKOSPI_Close','NASDAQ_return(%)']

In [18]:

# ============================================================
# #1. 하드보팅 함수
# ============================================================

def hard_voting_feature_selection(
    step_features,
    mul_features,
    boruta_features,
    target_col='Risk_Label',
    vote_threshold=2
):

    selection_dict = {
        'step': step_features,
        'mul': mul_features,
        'boruta': boruta_features
    }

    # 타겟 제거 + 중복 제거
    cleaned = {}
    for method, features in selection_dict.items():
        cleaned[method] = list(dict.fromkeys([
            f for f in features 
            if f != target_col
        ]))

    # 전체 변수 union
    all_features = sorted(set().union(*cleaned.values()))

    # 투표 테이블 생성
    rows = []
    for feature in all_features:
        selected_by = []
        vote_count = 0

        for method, features in cleaned.items():
            is_selected = feature in features
            if is_selected:
                vote_count += 1
                selected_by.append(method)

        rows.append({
            'feature': feature,
            'vote_count': vote_count,
            'selected_by_step': feature in cleaned['step'],
            'selected_by_mul': feature in cleaned['mul'],
            'selected_by_boruta': feature in cleaned['boruta'],
            'selected_by': ', '.join(selected_by)
        })

    vote_table = pd.DataFrame(rows)

    # 보기 좋게 정렬
    vote_table = vote_table.sort_values(
        by=['vote_count', 'feature'],
        ascending=[False, True]
    ).reset_index(drop=True)

    # 최종 선택 변수
    selected_features = vote_table.loc[
        vote_table['vote_count'] >= vote_threshold,
        'feature'
    ].tolist()

    return selected_features, vote_table


# ============================================================
# #2. 보팅 실행
# ============================================================

selected_features_vote, vote_table = hard_voting_feature_selection(
    step_features=step,
    mul_features=mul,
    boruta_features=boruta,
    target_col='Risk_Label',
    vote_threshold=2
)

print("=" * 80)
print("최종 선택 변수 개수:", len(selected_features_vote))
print("=" * 80)

for i, col in enumerate(selected_features_vote, 1):
    print(f"{i}. {col}")

print("\n" + "=" * 80)
print("전체 투표 결과")
print("=" * 80)

display(vote_table)


# ============================================================
# #3. 최종 모델링용 컬럼 생성
# ============================================================

final_cols = selected_features_vote + ['Risk_Label']

print("\n최종 사용 컬럼:")
print(final_cols)

최종 선택 변수 개수: 14
1. NASDAQ_return(%)
2. Brent Crude Oil_return(%)
3. Gold Spot_return(%)
4. KOSPI 200 Volume
5. KOSPI 200 lagged_1_return(%)
6. KOSPI 200_BB_width
7. KOSPI 200_OG
8. KOSPI 200_PPO_Hist
9. Signal4_Buy
10. Signal4_Sell
11. USD/KRW_change(%)
12. VKOSPI_Change(%)
13. VKOSPI_Close
14. return(%)

전체 투표 결과


,feature,vote_count,selected_by_step,selected_by_mul,selected_by_boruta,selected_by
0,NASDAQ_return(%),3,True,True,True,"step, mul, boruta"
1,Brent Crude Oil_return(%),2,False,True,True,"mul, boruta"
2,Gold Spot_return(%),2,True,True,False,"step, mul"
3,KOSPI 200 Volume,2,True,False,True,"step, boruta"
4,KOSPI 200 lagged_1_return(%),2,True,True,False,"step, mul"
5,KOSPI 200_BB_width,2,True,True,False,"step, mul"
6,KOSPI 200_OG,2,True,False,True,"step, boruta"
7,KOSPI 200_PPO_Hist,2,True,False,True,"step, boruta"
8,Signal4_Buy,2,True,True,False,"step, mul"
9,Signal4_Sell,2,True,True,False,"step, mul"



최종 사용 컬럼:
['NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'KOSPI 200 Volume', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_OG', 'KOSPI 200_PPO_Hist', 'Signal4_Buy', 'Signal4_Sell', 'USD/KRW_change(%)', 'VKOSPI_Change(%)', 'VKOSPI_Close', 'return(%)', 'Risk_Label']


In [19]:
# Date + 최종 선택 변수 + Risk_Label 저장
final_cols_with_date = ["Date"] + final_cols

# 중복 제거
final_cols_with_date = list(dict.fromkeys(final_cols_with_date))

# 실제 존재 컬럼 확인
missing_cols = [col for col in final_cols_with_date if col not in data.columns]
assert len(missing_cols) == 0, f"없는 컬럼 있음: {missing_cols}"

data[final_cols_with_date].to_csv(
    r"../../data/processed/data_selected.csv",
    index=False,
    encoding="utf-8-sig"
)

data[final_cols_with_date].head()

,Date,NASDAQ_return(%),Brent Crude Oil_return(%),Gold Spot_return(%),KOSPI 200 Volume,KOSPI 200 lagged_1_return(%),KOSPI 200_BB_width,KOSPI 200_OG,KOSPI 200_PPO_Hist,Signal4_Buy,Signal4_Sell,USD/KRW_change(%),VKOSPI_Change(%),VKOSPI_Close,return(%),Risk_Label
0,2009-04-17,0.157320,0.0,-1.362586,251600000.0,0.303254,0.131276,1.339314,0.132689,0,0,-0.549095,-2.10,35.49,-0.233192,0
1,2009-04-20,-3.953850,0.0,2.234472,183300000.0,-0.233192,0.131521,0.712077,0.056162,0,0,0.128139,1.86,36.15,0.564563,0
2,2009-04-21,2.191930,0.0,-0.553958,204200000.0,0.564563,0.112261,-2.159026,-0.040207,0,0,1.998728,0.69,36.40,-0.197523,0
3,2009-04-22,0.137996,0.0,1.093648,291400000.0,-0.197523,0.090814,0.767616,-0.038741,0,0,-0.570187,-3.82,35.01,1.408955,0
4,2009-04-23,0.369276,0.0,1.568707,277400000.0,1.408955,0.079146,0.848630,-0.003434,0,0,-0.970084,-4.63,33.39,0.992765,0
